# Analyse d'une campagne de marketing bancaire

## Objectif
Ce projet analyse les résultats d'une campagne de télémarketing menée par une banque portugaise,
afin d'identifier **quels profils de clients souscrivent le plus à un dépôt à terme**
et comment optimiser les prochaines campagnes.

## Source des données
Moro, S., Cortez, P., Rita, P. (2014). *A data-driven approach to predict the success of bank telemarketing.*
Decision Support Systems. Données issues de l'UCI Machine Learning Repository (licence CC BY 4.0).
Fichier utilisé : `bank-additional-full.csv` (41 188 appels, 21 variables).

## Plan du notebook
1. Chargement et premier aperçu des données
2. Diagnostic de la qualité des données
3. Nettoyage
4. Vérification et export de la donnée propre

## Problématique
Quels profils de clients souscrivent le plus à un dépôt à terme, et comment cibler plus efficacement les prochains appels ?

## 1. Chargement et premier aperçu

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/bank-additional-full.csv", sep=";") # import du fichier de données

In [3]:
df.head()  # Affiche les 5 premières lignes du fichier

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [4]:
df.shape  # Affiche la forme (lignes, colonnes)

(41188, 21)

In [5]:
df.info()  # Affiche le type des colonnes

<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  str    
 2   marital         41188 non-null  str    
 3   education       41188 non-null  str    
 4   default         41188 non-null  str    
 5   housing         41188 non-null  str    
 6   loan            41188 non-null  str    
 7   contact         41188 non-null  str    
 8   month           41188 non-null  str    
 9   day_of_week     41188 non-null  str    
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  str    
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null  float64
 1

In [ ]:
df.describe() #  Affiche les moyenne, écart-type, valeurs minimum et maximum sur les valeurs numériques

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,41188.00000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000
mean,40.02406,258.285010,2.567593,962.475454,0.172963,0.081886,93.575664,-40.502600,3.621291,5167.035911
std,10.42125,259.279249,2.770014,186.910907,0.494901,1.570960,0.578840,4.628198,1.734447,72.251528
min,17.00000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.634000,4963.600000
25%,32.00000,102.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.344000,5099.100000
50%,38.00000,180.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.00000,319.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,98.00000,4918.000000,56.000000,999.000000,7.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


In [ ]:
df.describe(include="str") # Affiche les données texte

,job,marital,education,default,housing,loan,contact,month,day_of_week,poutcome,y
count,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188
unique,12,4,8,3,3,3,2,10,5,3,2
top,admin.,married,university.degree,no,yes,no,cellular,may,thu,nonexistent,no
freq,10422,24928,12168,32588,21576,33950,26144,13769,8623,35563,36548


### Observations

**Structure du jeu de données**  
- 41 188 lignes et 21 colonnes : 10 colonnes numériques et 11 colonnes de texte.  
- Aucune valeur manquante détectée par pandas (`non-null` partout), mais les valeurs manquantes pourraient être codées autrement, par exemple sous forme de texte (`unknown`) ou de valeur fictive (`999`).  

**Points suspects à vérifier**  
- `unknown` : cette modalité apparaît dans les colonnes `job`, `marital`, `education`, `default`, `housing` et `loan` (3 modalités pour des questions oui/non). Il faut compter les `unknown` pour connaître exactement leur quantité par colonne.  
- `pdays` : la valeur 999 apparaît dès le quartile à 25 %, donc elle concernerait plus de 75 % des lignes. Il s'agirait d'un code signifiant « client jamais contacté auparavant » et non d'un nombre de jours réel. La moyenne (962) est donc trompeuse.  
- `poutcome` : la modalité `nonexistent` représente environ 86 % des lignes (35 563 sur 41 188), ce qui est cohérent avec `pdays = 999`. À confirmer en croisant les deux colonnes.  
- `duration` : le minimum est de 0 seconde et le maximum de 4 918 secondes (plus d'1 h 20), pour une médiane de 180 secondes. Un appel de 0 seconde correspond à une absence de conversation, et les appels très longs sont des valeurs extrêmes à examiner. Cette variable n'est connue qu'après l'appel : elle ne devra pas servir à expliquer ou prédire la souscription à l'avance.  
- `campaign` : la médiane est de 2 contacts mais le maximum atteint 56 appels pour un même client. Ces valeurs extrêmes sont à visualiser (histogramme ou boîte à moustaches) avant de décider quoi en faire.  
- Noms de colonnes : certains contiennent des points (`emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `nr.employed`), ce qui complique leur manipulation dans le code. Il faudra les renommer avec des underscores.  
- Variable cible `y` : elle est codée en texte (`yes` / `no`) et déséquilibrée, avec environ 88,7 % de `no` (36 548 clients). Il faudra la convertir en 0/1 pour calculer des taux de souscription.  

**Prochaine étape**  
Vérifier chaque suspicion par un chiffre dans le diagnostic : doublons, nombre de `unknown` par colonne, répartition de `y`, nombre de lignes avec `duration == 0` et `pdays == 999`.  


## 2. Diagnostic de la qualité

In [10]:
df.duplicated().sum() # Vérification des doublons

np.int64(12)

In [ ]:
(df == "unknown").sum() # Regarde combien de valeurs "unknown" sont présente dans le dataframe

age                  0
job                330
marital             80
education         1731
default           8597
housing            990
loan               990
contact              0
month                0
day_of_week          0
duration             0
campaign             0
pdays                0
previous             0
poutcome             0
emp.var.rate         0
cons.price.idx       0
cons.conf.idx        0
euribor3m            0
nr.employed          0
y                    0
dtype: int64

In [12]:
df["y"].value_counts(normalize=True) # Observe la répartition de "Y"

y
no     0.887346
yes    0.112654
Name: proportion, dtype: float64

In [15]:
(df["duration"] == 0).sum() # indique combien de lignes de la colonne "duration" sont égal a 0

np.int64(4)

In [17]:
(df["pdays"] == 999).sum() # indique combien de lignes "pdays" sont égales à 999

np.int64(39673)

In [18]:
pd.crosstab(df["pdays"] == 999, df["poutcome"]) # vérifie la cohérence entre pdays et poutcome

poutcome,failure,nonexistent,success
pdays,,,
False,142,0,1373
True,4110,35563,0


In [19]:
(df == "unknown").sum() / len(df) * 100 # transforme les comptages de unknown en pourcentages

age                0.000000
job                0.801204
marital            0.194231
education          4.202680
default           20.872584
housing            2.403613
loan               2.403613
contact            0.000000
month              0.000000
day_of_week        0.000000
duration           0.000000
campaign           0.000000
pdays              0.000000
previous           0.000000
poutcome           0.000000
emp.var.rate       0.000000
cons.price.idx     0.000000
cons.conf.idx      0.000000
euribor3m          0.000000
nr.employed        0.000000
y                  0.000000
dtype: float64

In [20]:
pd.crosstab(df["default"], df["y"], normalize="index") * 100 # Affiche le taux de souscription par modalité de default

y,no,yes
default,,
no,87.121026,12.878974
unknown,94.847040,5.152960
yes,100.000000,0.000000


In [21]:
df["default"].value_counts() # compte combien de fois chaque valeur apparaît dans la colonne default

default
no         32588
unknown     8597
yes            3
Name: count, dtype: int64

In [22]:
pd.crosstab(df["education"], df["y"], normalize="index") * 100 # croise deux colonnes et affiche, pour chaque niveau d'éducation, le pourcentage de clients qui ont dit no et yes

y,no,yes
education,,
basic.4y,89.750958,10.249042
basic.6y,91.797557,8.202443
basic.9y,92.175352,7.824648
high.school,89.164477,10.835523
illiterate,77.777778,22.222222
professional.course,88.651535,11.348465
university.degree,86.275477,13.724523
unknown,85.499711,14.500289


In [23]:
((df["housing"] == "unknown") & (df["loan"] == "unknown")).sum() # vérifie si les 990 unknown de housing et loan sont les mêmes lignes

np.int64(990)

In [24]:
df["education"].value_counts() # compte combien de fois chaque valeur apparaît dans la colonne education

education
university.degree      12168
high.school             9515
basic.9y                6045
professional.course     5243
basic.4y                4176
basic.6y                2292
unknown                 1731
illiterate                18
Name: count, dtype: int64

In [ ]:
pd.crosstab(df["job"], df["y"], normalize="index") * 100 # croise deux colonnes et affiche, pour chaque niveau de job, le pourcentage de clients qui ont dit no et yes

y,no,yes
job,,
admin.,87.027442,12.972558
blue-collar,93.105684,6.894316
entrepreneur,91.483516,8.516484
housemaid,90.000000,10.000000
management,88.782490,11.217510
retired,74.767442,25.232558
self-employed,89.514426,10.485574
services,91.861930,8.138070
student,68.571429,31.428571


In [26]:
pd.crosstab(df["marital"], df["y"], normalize="index") * 100 # croise deux colonnes et affiche, pour chaque niveau de marital, le pourcentage de clients qui ont dit no et yes

y,no,yes
marital,,
divorced,89.679098,10.320902
married,89.842747,10.157253
single,85.995851,14.004149
unknown,85.000000,15.000000


In [27]:
pd.crosstab(df["housing"], df["y"], normalize="index") * 100 # croise deux colonnes et affiche, pour chaque niveau de housing, le pourcentage de clients qui ont dit no et yes

y,no,yes
housing,,
no,89.120395,10.879605
unknown,89.191919,10.808081
yes,88.380608,11.619392


In [28]:
pd.crosstab(df["loan"], df["y"], normalize="index") * 100 # croise deux colonnes et affiche, pour chaque niveau de loan, le pourcentage de clients qui ont dit no et yes

y,no,yes
loan,,
no,88.659794,11.340206
unknown,89.191919,10.808081
yes,89.068502,10.931498


In [29]:
df[df["duration"] == 0] # vérifie ce que vaut "y" pour les 4 lignes où duration = 0

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
6251,39,admin.,married,high.school,no,yes,no,telephone,may,tue,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
23031,59,management,married,university.degree,no,yes,no,cellular,aug,tue,...,10,999,0,nonexistent,1.4,93.444,-36.1,4.965,5228.1,no
28063,53,blue-collar,divorced,high.school,no,yes,no,cellular,apr,fri,...,3,999,0,nonexistent,-1.8,93.075,-47.1,1.479,5099.1,no
33015,31,blue-collar,married,basic.9y,no,no,no,cellular,may,mon,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.299,5099.1,no


### Observations du diagnostic

**Doublons**  
- 12 lignes strictement identiques sur 41 188 (0,03 %).  

**Valeurs `unknown`**  
- Colonnes concernées : `default` (8 597 ; 20,9 %), `education` (1 731 ; 4,2 %), `housing` et `loan` (990 chacune ; 2,4 %), `job` (330 ; 0,8 %), `marital` (80 ; 0,2 %).  
- Les 990 `unknown` de `housing` et `loan` correspondent aux **mêmes clients**.  
- Taux de souscription des `unknown` comparés à la moyenne (11,3 %) :  
  - `default` : 5,2 % contre 12,9 % pour les `no`. L'absence d'information est un signal.  
  - `education` : 14,5 %, supérieur à tous les niveaux principaux. Les `unknown` ne ressemblent à aucun niveau précis.  
  - `job`, `housing`, `loan` : environ 11 %, soit la moyenne. Les `unknown` sont neutres.  
  - `marital` : 80 clients seulement, trop peu pour interpréter.  
- Attention à la taille des groupes : `default = yes` (3 clients) et `education = illiterate` (18 clients) sont trop petits pour que leurs taux soient interprétables.  

**Variable cible `y`**  
- 88,7 % de `no` et 11,3 % de `yes` : classes très déséquilibrées. Le taux de souscription de référence est d'environ 11,3 %.  

**`duration`**  
- 4 lignes à 0 seconde, toutes avec `y = no`. Sans conversation, ce `no` n'est pas un vrai refus.  

**`pdays`**  
- 39 673 lignes (96,3 %) valent 999. Ce n'est pas un délai réel mais un code.  
- Le croisement avec `poutcome` montre que 4 110 de ces clients ont `poutcome = failure` : ils ont déjà été contactés, mais le délai n'est pas connu. `pdays = 999` ne signifie donc pas toujours « jamais contacté ». Pour savoir si un client a déjà été contacté, `poutcome` est plus fiable.  

**Prochaine étape**
Section 3 : décider du traitement de chaque point, puis l'appliquer.  

## 3. Nettoyage

### Décisions

| Sujet | Décision | Justification |
|---|---|---|
| `unknown` (`default`, `education`, `job`, `marital`, `housing`, `loan`) | Conservés comme modalité à part | Aucune perte de données. Pour `default` et `education`, les `unknown` ont un taux de souscription différent de la moyenne, ils portent donc une information. Pour `job`, `housing` et `loan`, ils sont neutres (taux proche de 11 %) et ne biaisent rien. Pour `marital`, le groupe est trop petit (80 lignes) pour avoir un impact. Une règle unique pour toutes les colonnes. |
| Doublons (12 lignes) | Supprimés | Lignes strictement identiques sur les 21 colonnes : très probablement des erreurs de saisie ou d'extraction. Les garder compterait deux fois le même appel. Impact négligeable (0,03 %). |
| `duration == 0` (4 lignes) | Supprimés | Aucune conversation n'a eu lieu, donc le `no` n'est pas un vrai refus. Ces lignes n'apportent aucune information sur la réaction du client. Impact négligeable (0,01 %). |
| `pdays == 999` (39 673 lignes) | Remplacé par `NaN` | 999 est un code (« délai inconnu ou sans objet ») et non un nombre de jours. Le laisser fausserait les moyennes (962 jours). Avec `NaN`, pandas ignore ces lignes dans les calculs. Pour savoir si un client a déjà été contacté, on s'appuiera sur `poutcome`. |
| Noms de colonnes avec des points (`emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `nr.employed`) | Renommés avec des underscores | Plus simples à manipuler dans le code, les graphiques et Streamlit (le point peut être confondu avec un accès à un attribut). |
| `y` (`yes` / `no`) | Convertie en 0 / 1 | Permet de calculer directement un taux de souscription : la moyenne de la colonne. |